# 住民の視覚パイプライン解説 — レイキャスト → DINOv2 → CLIP

**1体の住民が、1つの場面で「何を見て、どう判断しているか」を最初から最後まで追う。**

数字の検証は S0-4 / S0-5 / S0-6 で済んでいる。このノートはその結果を前提に、
**仕組みそのものを可視化して理解する**ことが目的。実行するたびに新しい場面が
選ばれるので、毎回違う建物・違う結果になる。

---

### 全体像

```
① レイキャスト    224本のレイで画像を作る (テクスチャを貼り、距離を明るさ/高さに変換)
                     │
        ┌────────────┴────────────┐
        │                          │
② DINOv2 (現状で実際に使われている) │
   → 384次元に要約 → 方策へ          ③ 外の世界向け拡張 (perception.py)
   建物の種別はここでは分からない       DepthAnything で距離を推定
                                     → 境界検出で建物ごとに切り分け
                                     → CLIP でグループを推定
                                     → 訪問したら記憶で確定
```

**現状のシミュレーション内部では、建物の種別はマップ配列から直接引いている。**
DINOv2 も CLIP も、種別の判定には使われていない。③ は「もし配列が無い外の世界に
出たら、視覚だけでどこまで代替できるか」を検証した拡張で、`mesa_env.perception`
として実装済み。このノートはその両方を、同じ場面に対して並べて見せる。

---

### どこで、何の精度を見ているか

| 段階 | 見ているもの | 意味 |
|---|---|---|
| ② DINOv2 | 特徴ベクトルの中身 | 建物種別の情報を持たないことを直接確認する |
| ③-深度 | DepthAnything vs 正解距離 の相関 | 「壁までの距離」を画像だけから当てられるか |
| ③-境界 | 検出した区切りの位置 vs 正解の区切り | 「どこからどこまでが1棟か」を切り分けられるか |
| ③-分類 | CLIP の Top-5 vs 正解タイプ・正解グループ | 「何の建物か」をどこまで当てられるか |
| ③-記憶 | 複数回の観測の投票 → 訪問による確定 | 1回の誤判定に引きずられない仕組み |

---

左のファイルペインに `mesa-env.zip` をアップロードしてから実行。GPU 推奨。

In [ ]:
!pip install -q transformers accelerate open_clip_torch japanize-matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/mesa_textures', exist_ok=True)
os.system('cp -r /content/drive/MyDrive/mesa_textures/. /content/mesa_textures/')
print(sorted(os.listdir('/content/mesa_textures'))[:10])

In [ ]:
import os, sys, glob, zipfile, importlib
if not glob.glob('**/mesa_env/__init__.py', recursive=True):
    zips = glob.glob('**/mesa-env.zip', recursive=True)
    if not zips:
        raise FileNotFoundError('mesa-env.zip を左のフォルダにアップロードしてください')
    zipfile.ZipFile(zips[0]).extractall('.')
ROOT = os.path.abspath(os.path.dirname(os.path.dirname(
    glob.glob('**/mesa_env/__init__.py', recursive=True)[0])))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
importlib.invalidate_caches()

import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import japanize_matplotlib   # matplotlib で日本語が豆腐(□)にならないようにする

from mesa_env import (ALIGNED, Raycaster, load_texture_bank, BuildingMemory,
                      PerceptionPipeline, detect_boundaries, segments_from_boundaries,
                      guess_groups, group_of, GROUP_NAMES, SEMANTIC_GROUPS)
from mesa_env.maps import build_map
from mesa_env.render import TREE_TEX_INDEX
from mesa_env.world import WorldConfig
from mesa_env.constants import BLDG_TYPE_NAMES, GRID

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TEX_DIR = '/content/mesa_textures'
bank, counts = load_texture_bank(TEX_DIR, device=DEVICE)
rc = Raycaster(bank, device=DEVICE)
WORLD = WorldConfig(solid_buildings=True, visible_trees=True, walkable_empty=True,
                    building_heights=True, visible_agents=False)
print(f'mesa-env OK  device={DEVICE}')

## モデルの読み込み

3つのモデルを使う。役割がそれぞれ違うことに注意。

- **DINOv2** — 現状、方策(行動を決めるネットワーク)が実際に見ている特徴量。凍結。
- **DepthAnything** — 外の世界向け拡張。1枚の画像から距離を推定する (単眼深度推定)
- **CLIP** — 外の世界向け拡張。画像とテキストを同じ空間に埋め込む。建物の意味判定に使う

In [ ]:
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(DEVICE).eval()
for p in dino.parameters():
    p.requires_grad_(False)
DINO_MEAN = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
DINO_STD  = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)

@torch.no_grad()
def dino_forward(img_chw_batch):
    """(N,3,H,W)[0,1] -> dict(cls=(N,384), patch=(N,256,384))"""
    x = (F.interpolate(img_chw_batch, size=(224,224), mode='bilinear', align_corners=False)
         .to(DEVICE) - DINO_MEAN) / DINO_STD
    out = dino.forward_features(x)
    return dict(cls=out['x_norm_clstoken'], patch=out['x_norm_patchtokens'])

print('DINOv2 ViT-S/14 読み込み完了')

In [ ]:
from transformers import pipeline as hf_pipeline
DEPTH_MODEL = 'depth-anything/Depth-Anything-V2-Small-hf'
depth_pipe = hf_pipeline(task='depth-estimation', model=DEPTH_MODEL,
                         device=0 if DEVICE == 'cuda' else -1)

@torch.no_grad()
def predict_depth(img_chw: torch.Tensor) -> np.ndarray:
    from PIL import Image
    arr = (img_chw.clamp(0,1).permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
    out = depth_pipe(Image.fromarray(arr))
    dm = np.asarray(out['predicted_depth'].squeeze().cpu())
    return torch.nn.functional.interpolate(
        torch.tensor(dm)[None,None].float(), size=(rc.H, rc.W),
        mode='bilinear', align_corners=False)[0,0].numpy()

print(f'DepthAnything: {DEPTH_MODEL}')

In [ ]:
import open_clip

LABEL_TEXT = {
    'kiosk':'a small street food kiosk', 'conbini':'a Japanese convenience store',
    'pharmacy':'a drugstore pharmacy', 'cafe':'a coffee shop cafe',
    'gyudon':'a Japanese beef bowl fast food restaurant',
    'ramen':'a Japanese ramen noodle restaurant',
    'bento':'a Japanese bento lunch box takeaway shop',
    'shop':'a small retail shop storefront', 'house':'a detached suburban house',
    'post':'a post office', 'bank':'a bank branch building',
    'apartment':'an apartment building', 'hotel':'a hotel building',
    'office':'an office building', 'tower':'a high rise tower',
    'supermarket':'a supermarket', 'temple':'a Japanese temple shrine',
    'school':'a school building', 'station':'a train station',
    'library':'a public library', 'hospital':'a hospital',
    'cityhall':'a city hall government building', 'museum':'a museum',
    'stadium':'a sports stadium', 'mall':'a shopping mall',
}
TEMPLATES = ['a photo of {}.', 'a photo of the storefront of {}.',
             'a street view photo of {}.', 'the exterior facade of {}.']
CLIP_NAME, CLIP_PRETRAINED = 'ViT-B-32', 'laion2b_s34b_b79k'
clip_model, _, _ = open_clip.create_model_and_transforms(CLIP_NAME, pretrained=CLIP_PRETRAINED)
clip_model = clip_model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(CLIP_NAME)
CM = torch.tensor([0.48145466,0.4578275,0.40821073]).view(1,3,1,1).to(DEVICE)
CS = torch.tensor([0.26862954,0.26130258,0.27577711]).view(1,3,1,1).to(DEVICE)

@torch.no_grad()
def clip_encode_images(x):
    b = F.interpolate(x.to(DEVICE), size=(224,224), mode='bilinear', align_corners=False)
    return F.normalize(clip_model.encode_image((b-CM)/CS).float(), dim=-1).cpu()

@torch.no_grad()
def clip_encode_text(descs):
    out = []
    for v in descs:
        e = clip_model.encode_text(tokenizer([t.format(v) for t in TEMPLATES]).to(DEVICE)).float()
        out.append(F.normalize(F.normalize(e, dim=-1).mean(0), dim=-1).cpu())
    return torch.stack(out)

TXT = clip_encode_text([LABEL_TEXT[n] for n in BLDG_TYPE_NAMES])
print(f'CLIP {CLIP_NAME}/{CLIP_PRETRAINED}  準備完了')

## 図1: 全体像を描く

In [ ]:

# ── 図1: パイプライン全体図 ──
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set_xlim(0, 11); ax.set_ylim(0, 4.5); ax.axis('off')

def box(x, y, w, h, text, color):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=color, ec='#333', lw=1))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=9.5)

def arrow(x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', lw=1.4, color='#333'))

box(0.3, 2.6, 2.4, 1.3, '① レイキャスト\n224本のレイで\n一人称画像を作る', '#cfe8ff')
box(3.7, 3.3, 2.6, 1.0, '② DINOv2\n(現状使われている経路)\n384次元 → 方策へ', '#d7f5d7')
box(3.7, 1.1, 2.6, 1.0, '③ DepthAnything\n距離を推定', '#ffe6cc')
box(7.1, 1.1, 1.9, 1.0, '境界検出', '#ffe6cc')
box(9.7, 1.1, 1.1, 1.0, 'CLIP', '#ffe6cc')
arrow(2.7, 3.6, 3.7, 3.8)
arrow(2.7, 3.0, 3.7, 1.6)
arrow(6.3, 1.6, 7.1, 1.6)
arrow(9.0, 1.6, 9.7, 1.6)
ax.text(0.3, 0.3, '②は常時使われている経路。③は「配列が無い外の世界」向けの拡張プロトタイプ。',
       fontsize=9, color='#555')
plt.tight_layout(); plt.savefig('/content/fig01_overview.png', dpi=140, bbox_inches='tight')
plt.show()


---
## STEP 0 — 住民から見えている景色

ランダムな場所・向きに住民を1人置いて、実際に見えている一人称画像を作る。
**これが以降すべてのステップの入力になる、たった1枚の画像。**

In [ ]:
rng = np.random.default_rng()
spec = build_map(seed=int(rng.integers(0, 1_000_000)), world=WORLD)
pos = np.argwhere(spec.passable)

# 中央のレイが建物に当たる場所を選ぶ (何も映っていない場面だと解説にならないため)
for _ in range(200):
    pk = rng.integers(0, len(pos))
    x0 = torch.tensor([pos[pk,0] + rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    y0 = torch.tensor([pos[pk,1] + rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    th0 = torch.tensor([rng.uniform(0, 2*np.pi)], dtype=torch.float32, device=DEVICE)
    perp0, hitT0, *_ = rc.cast(x0, y0, th0, spec)
    if 0 <= hitT0[0, rc.W//2] < len(BLDG_TYPE_NAMES) and perp0[0, rc.W//2] < 7.0:
        break

scene_img = rc(x0, y0, th0, spec)[0].cpu()    # (3,H,W) ← .cpu() で以降の可視化を安全にする
scene_hitT = hitT0[0].cpu().numpy()           # (W,) 正解: 各列が何に当たったか
scene_perp = perp0[0].cpu().numpy()           # (W,) 正解: 各列までの距離

plt.figure(figsize=(9,9))
plt.imshow(scene_img.permute(1,2,0).clamp(0,1).numpy())
plt.axis('off')
plt.title('住民の一人称視点 (この1枚を、これから3つの方法で解析する)')
plt.show()

n_bldg = (scene_hitT >= 0) & (scene_hitT < len(BLDG_TYPE_NAMES))
print(f'視界に映っている建物の列: {int(n_bldg.sum())}/{rc.W} 列')
print(f'木が映っている列: {int((scene_hitT==TREE_TEX_INDEX).sum())}/{rc.W} 列')

In [ ]:

# ── 図2: 1本のレイがテクスチャの1列を画面に貼る様子 ──
from mesa_env.render import TEX_SIZE

# 視界内で実際に使われたテクスチャを1枚拾う (STEP0のsceneから)
perp_s, hitT_s, hitV_s, *_ = rc.cast(x0, y0, th0, spec)
mid = rc.W // 2
t_idx, v_idx = int(hitT_s[0, mid]), int(hitV_s[0, mid])
tex = bank[t_idx, v_idx].cpu().numpy()          # (64,64,3) 実際に貼られたテクスチャ

# このレイが貼った列番号 (texXi 相当) を再現する簡易版
demo_col = TEX_SIZE // 2
column = tex[:, demo_col]                        # (64,3) テクスチャの1列
stretched = np.repeat(column[None, :, :], 160, axis=0)  # 縦に引き伸ばす (デモ用)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
axes[0].imshow(tex); axes[0].axvline(demo_col, color='red', lw=2)
axes[0].set_title(f'元テクスチャ ({BLDG_TYPE_NAMES[t_idx]})\n赤線=このレイが拾う列')
axes[1].imshow(stretched)
axes[1].set_title('その1列を、距離に応じて\n縦に引き伸ばす')
axes[2].imshow(scene_img.permute(1,2,0).clamp(0,1).numpy())
axes[2].axvline(mid, color='red', lw=2)
axes[2].set_title('224本ぶん並べると\n画像全体が復元される')
for a in axes: a.axis('off')
plt.tight_layout(); plt.savefig('/content/fig02_raycast_column.png', dpi=140, bbox_inches='tight')
plt.show()


---
## STEP 1 — レイキャストが作ったもの: 距離と種別 (正解)

この画像を作った時点で、**レイキャストは既に全ての答えを知っている。**
224本のレイそれぞれが「何に当たったか」「どれだけ離れているか」を持っている。
これは配列を直接引いた**グラウンドトゥルース**で、これから先のステップは
「もしこの答えを画像からしか得られなかったら」を試すもの。

In [ ]:
fig, ax = plt.subplots(figsize=(11,3))
type_row = np.where(scene_hitT==TREE_TEX_INDEX, 25, scene_hitT)
ax.plot(scene_perp, color='#2b6cb0', label='距離 perp (正解)')
ax2 = ax.twinx()
ax2.step(range(rc.W), type_row, color='#c05621', alpha=0.6, where='mid', label='当たったタイプ (正解)')
ax.set_xlabel('画面の列 (0〜223)'); ax.set_ylabel('距離 (セル)', color='#2b6cb0')
ax2.set_ylabel('タイプ index (25=木)', color='#c05621')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.title('224本のレイが持っている正解情報')
plt.tight_layout(); plt.savefig('/content/fig03_ground_truth_row.png', dpi=140, bbox_inches='tight')
plt.show()

---
## STEP 2 — DINOv2: 方策が実際に使っている特徴

**現状のシステムでは、ここが行動決定に使われている唯一の視覚経路。**
先に結論を言うと、DINOv2 は言葉を学んでいないので「これはラーメン屋だ」とは
分からない。384次元は「見た目の記述」であって「意味のラベル」ではない。

それを実際に確認する。patch token (画像を16×16のパッチに分けた、それぞれの
特徴) を3次元に圧縮して色として可視化する。**似た色 = DINOv2 にとって似ている
見た目。** テクスチャの模様や壁の輪郭には反応するが、店の種類という概念には
反応しないはず。

In [ ]:
feat = dino_forward(scene_img[None])
cls_vec = feat['cls'][0]              # (384,) 方策に渡る特徴そのもの
patch = feat['patch'][0]              # (256, 384) = 16x16 パッチ

print(f'CLS token: {cls_vec.shape}  ノルム {float(cls_vec.norm()):.1f}')
print(f'patch tokens: {patch.shape}  (16x16 グリッド, 1パッチ=14x14px)')

# patch を PCA で3次元に落として RGB として可視化 (DINOv2 の「注目箇所」の定番の見方)
pc = patch.cpu().numpy()
pc = pc - pc.mean(0, keepdims=True)
u, s, vt = np.linalg.svd(pc, full_matrices=False)
proj = (pc @ vt[:3].T)
proj = (proj - proj.min(0)) / (proj.max(0) - proj.min(0) + 1e-9)
pca_map = proj.reshape(16, 16, 3)

fig, axes = plt.subplots(1, 2, figsize=(14,7))
axes[0].imshow(scene_img.permute(1,2,0).clamp(0,1).numpy()); axes[0].set_title('元画像')
axes[1].imshow(pca_map); axes[1].set_title('DINOv2 patch token の PCA可視化\n(色=見た目の類似性。建物種別のラベルではない)')
for a in axes: a.axis('off')
plt.tight_layout(); plt.savefig('/content/fig04_dino_pca.png', dpi=140, bbox_inches='tight')
plt.show()

print('この384次元 (CLS) がそのまま方策ネットワークに入る。')
print('ただしナビゲーションの主判断 (どっちへ進む・ぶつからない) は compass と')
print('obstacle レイが担っており、視覚は「目的地が見えたら寄せる」程度の微調整。')

## 寄り道 — 「見えるもの」と「通れるもの」がズレていた話

STEP3 に進む前に、1つ根本的な問題を直しておく必要がありました。きっかけは
「レイの距離が分かるなら、3m以上なら進める、という単純なルールで障害物を
避けられないか」という素朴な疑問でした。

やってみると6回に1回は壁に突っ込むという結果になり、調べると MESA の世界は
「見えるもの」と「通れるもの」がほぼ逆転していました。建物は壁として画面に
映るのに実は通り抜けられ、木は画面に一切映らないのに実は通行不可でした。

これを `solid_buildings` / `visible_trees` / `walkable_empty` の3点で修正し、
相関が実際にどう改善するかを測ります。

In [ ]:

# ── 図5: 光学(見える)と物理(通れる)の相関を、3段階の設定で再計測する ──
from mesa_env.constants import BUILDING, ROAD, TREE, GRID
from mesa_env.world import WorldConfig
from scipy.stats import spearmanr as _sr

def _optics_physics_corr(world, n_trials=400, seed=0):
    sp = build_map(seed=seed, world=world)
    passable = sp.passable
    pos = np.argwhere(passable)
    rgen = np.random.default_rng(seed)
    pk = rgen.integers(0, len(pos), n_trials)
    xs = torch.tensor(pos[pk,0]+0.5, dtype=torch.float32, device=DEVICE)
    ys = torch.tensor(pos[pk,1]+0.5, dtype=torch.float32, device=DEVICE)
    ths = torch.tensor(rgen.uniform(0, 2*np.pi, n_trials), dtype=torch.float32, device=DEVICE)
    perp, *_ = rc.cast(xs, ys, ths, sp)
    ray_dist = perp[:, rc.W//2].cpu().numpy()          # レイが「見た」距離

    # 物理的に実際に何マス進めるかを、真の通行判定で愚直に測る (正解)
    true_dist = np.zeros(n_trials)
    cells = sp.cells
    for i in range(n_trials):
        cx, cy, th = float(xs[i]), float(ys[i]), float(ths[i])
        d = 0.0
        while d < 8.0:
            d += 0.1
            r, c = int(cx+np.cos(th)*d), int(cy+np.sin(th)*d)
            if not (0 <= r < GRID and 0 <= c < GRID) or not passable[r, c]:
                break
        true_dist[i] = d
    rho, _ = _sr(ray_dist, true_dist)
    return abs(rho)

configs = [
    ('修正前\n(現行デフォルト)', WorldConfig()),
    ('建物だけ\n通行不可に', WorldConfig(solid_buildings=True)),
    ('+ 木を可視化', WorldConfig(solid_buildings=True, visible_trees=True)),
    ('3点すべて修正\n(ALIGNED)', ALIGNED),
]
rhos = [_optics_physics_corr(w) for _, w in configs]

fig, ax = plt.subplots(figsize=(8,4.5))
bars = ax.bar([c[0] for c in configs], rhos, color=['#e07b7b','#e0b97b','#e0d87b','#7bc47b'])
for b, r in zip(bars, rhos):
    ax.text(b.get_x()+b.get_width()/2, r+0.02, f'{r:.2f}', ha='center')
ax.set_ylim(0, 1.1); ax.set_ylabel('レイの距離 と 実際に進める距離 の相関 |rho|')
ax.set_title('光学(見えるもの)と物理(通れるもの)を揃えていく過程')
plt.tight_layout(); plt.savefig('/content/fig05_optics_physics.png', dpi=140, bbox_inches='tight')
plt.show()
print(f'相関の推移: {[round(r,3) for r in rhos]}')


---
## STEP 3 — もし配列が無かったら: DepthAnything で距離を推定する

ここから「外の世界向け拡張」(`mesa_env.perception`)。**もう配列は見ない。**
画像だけから、STEP 1 の正解距離をどこまで当てられるか。

In [ ]:
HORIZON_ROW = rc.H // 2       # 目線の高さ。どの建物でも安定した比較基準
depth_map = predict_depth(scene_img)
depth_row = depth_map[HORIZON_ROW]

from scipy.stats import spearmanr
valid = scene_perp < 7.0
rho, _ = spearmanr(depth_row[valid], scene_perp[valid])

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].imshow(depth_map, cmap='inferno')
axes[0].axhline(HORIZON_ROW, color='cyan', lw=1)
axes[0].set_title('DepthAnything の予測深度マップ\n(水色の線=これから使う目線の高さ)')
axes[0].axis('off')

ax = axes[1]
ax.plot(scene_perp, color='#2b6cb0', label='正解 (perp)')
ax2 = ax.twinx()
ax2.plot(depth_row, color='#dd6b20', label='予測 (DepthAnything)')
ax.set_xlabel('画面の列'); ax.set_ylabel('正解距離', color='#2b6cb0')
ax2.set_ylabel('予測深度 (相対値)', color='#dd6b20')
ax.set_title(f'この場面での順位相関 |rho| = {abs(rho):.3f}')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout(); plt.savefig('/content/fig06_depth_comparison.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'見ている精度: 「近い/遠いの順序」がどれだけ合っているか (順位相関)')
print(f'この1場面: rho = {rho:+.3f}   (複数場面での実測中央値は 0.997 — 非常に高精度)')
print('DepthAnything の値は絶対距離ではなく相対値なので、向き(符号)はモデル依存。')

---
## STEP 4 — 境界検出: どこからどこまでが1棟か

距離が急に変わる場所 = 手前の建物が終わって、奥の建物が始まる場所、とみなす。
`mesa_env.perception.detect_boundaries` を使う。

**見ている精度:** 検出した境界の位置が、正解の境界 (STEP1で分かっている
`hitT` の変化点) とどれだけ合っているか。

In [ ]:
boundaries = detect_boundaries(depth_row)
segs = segments_from_boundaries(boundaries, rc.W)
true_boundaries = np.where(np.diff(scene_hitT) != 0)[0]

fig, ax = plt.subplots(figsize=(11,7))
ax.imshow(scene_img.permute(1,2,0).clamp(0,1).numpy())
for b in true_boundaries:
    ax.axvline(b, color='lime', lw=1.5, alpha=0.8)
for b in boundaries:
    ax.axvline(b, color='red', lw=1.5, linestyle='--', alpha=0.8)
ax.legend(handles=[mpatches.Patch(color='lime', label='正解の境界'),
                   mpatches.Patch(color='red', label='検出した境界 (推定)')],
          loc='upper right')
ax.set_title(f'検出された建物の区切り: {len(segs)}個のセグメント')
ax.axis('off')
plt.tight_layout(); plt.show()

print(f'検出したセグメント (列範囲): {segs}')
print(f'正解の境界位置: {true_boundaries.tolist()}')

In [ ]:

# ── 図7・8: 境界検出。壊れていた旧実装(quantile)と、現行の修正版を並べて比較する ──
def _old_boundaries(depth_row, q=0.9):
    """以前使っていた実装。画像の建物数と無関係に常に上位10%を境界とみなしてしまう。"""
    g = np.abs(np.diff(depth_row))
    thr = np.quantile(g, q)
    return np.where(g > thr)[0]

old_b = _old_boundaries(depth_row)
new_b = detect_boundaries(depth_row)          # mesa_env.perception の現行実装
true_b = true_boundaries                       # 正解 (STEP4 セルで既に計算済み)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (bnds, title) in zip(axes, [(old_b, f'旧実装 (quantile)\n検出数={len(old_b)}件'),
                                    (new_b, f'現行実装 (ロバストz-score)\n検出数={len(new_b)}件')]):
    ax.imshow(scene_img.permute(1,2,0).clamp(0,1).numpy())
    for b in true_b:
        ax.axvline(b, color='lime', lw=1.5, alpha=0.85)
    for b in bnds:
        ax.axvline(b, color='red', lw=1.3, linestyle='--', alpha=0.8)
    ax.set_title(title); ax.axis('off')
axes[0].legend(handles=[mpatches.Patch(color='lime', label='正解の境界'),
                        mpatches.Patch(color='red', label='検出した境界')], loc='upper right')
plt.tight_layout(); plt.savefig('/content/fig07_08_boundary_compare.png', dpi=140, bbox_inches='tight')
plt.show()
print(f'旧実装: {len(old_b)}件検出 (建物数と無関係に一定割合を検出してしまう)')
print(f'現行実装: {len(new_b)}件検出')


---
## STEP 5 — CLIP: 「何の建物か」を当てる

各セグメントを切り出して CLIP に見せる。**個別のタイプを一発で当てるのは
実測で Top-5=0.39 と難しい。** ただし「食べ物系」「大型ビル系」といった
**グループ**まで丸めると Top-5=0.70 まで上がる。ここではその両方を、
このシーンの実際の建物で見る。

In [ ]:
def segment_label(hitT_seg):
    vals, cnt = np.unique(hitT_seg, return_counts=True)
    ok = (vals >= 0) & (vals < len(BLDG_TYPE_NAMES))
    if not ok.any():
        return None
    vals, cnt = vals[ok], cnt[ok]
    return BLDG_TYPE_NAMES[int(vals[np.argmax(cnt)])]

fig, axes = plt.subplots(1, max(len(segs),1), figsize=(4.2*max(len(segs),1), 4.6))
if len(segs) == 1:
    axes = [axes]

for ax, (c0, c1) in zip(axes, segs):
    true_name = segment_label(scene_hitT[c0:c1])
    crop = scene_img[:, :, c0:c1]
    ax.imshow(crop.permute(1,2,0).clamp(0,1).numpy())
    ax.axis('off')
    if true_name is None:
        ax.set_title('(木・空き地)', fontsize=10)
        continue

    emb = clip_encode_images(crop[None])[0]
    guess = guess_groups(emb, TXT, k=5)
    true_group = group_of(true_name)
    ok_type = guess.top_type == true_name
    ok_group = true_group in guess.candidate_groups

    logits = emb @ TXT.T
    top5_idx = logits.topk(5).indices.tolist()
    top5_str = '\n'.join(f'  {BLDG_TYPE_NAMES[i]} ({float(logits[i]):.2f})' for i in top5_idx)

    print(f'--- セグメント [{c0}:{c1}] 正解={true_name} ({true_group}) ---')
    print(f'CLIP Top-5:\n{top5_str}')
    print(f'  タイプ的中: {"O" if ok_type else "X"}   グループ的中: {"O" if ok_group else "X"} '
          f'(候補グループ: {guess.candidate_groups})\n')

    mark = f"{'O' if ok_group else 'X'}"
    ax.set_title(f'正解:{true_name}\nCLIP:{guess.top_type} [{mark}]', fontsize=10)

plt.tight_layout(); plt.show()

### 図9: 過去に踏んだバグを再現してみる

In [ ]:

# ── 図9: 過去に踏んだバグの再現デモ ──
# 木 (TREE_TEX_INDEX) を正解ラベルの多数決から除外し忘れると何が起きるか、
# あえて当時のバグ入りコードで再現する。実際に踏んだバグの記録として。
def _segment_label_buggy(hitT_seg):
    """バグ版: 木 (index=25) も "有効なタイプ" として通してしまう。"""
    vals, cnt = np.unique(hitT_seg[hitT_seg >= 0], return_counts=True)   # ← >=0 だけの判定
    return int(vals[np.argmax(cnt)]) if len(vals) else -1

fig, axes = plt.subplots(1, min(len(segs), 5), figsize=(4.2*min(len(segs),5), 4.6))
if min(len(segs), 5) == 1:
    axes = [axes]

for ax, (c0, c1) in zip(axes, segs[:5]):
    buggy_label_idx = _segment_label_buggy(scene_hitT[c0:c1])
    crop = scene_img[:, :, c0:c1]
    ax.imshow(crop.permute(1,2,0).clamp(0,1).numpy())
    ax.axis('off')
    if buggy_label_idx == TREE_TEX_INDEX:
        ax.set_title('正解ラベル = 木(25)\n→ CLIPが何を答えても\n絶対に当たらない', fontsize=9, color='red')
    elif 0 <= buggy_label_idx < len(BLDG_TYPE_NAMES):
        ax.set_title(f'正解:{BLDG_TYPE_NAMES[buggy_label_idx]}\n(このセグメントは正常)', fontsize=9)
    else:
        ax.set_title('(該当なし)', fontsize=9)

plt.suptitle('バグ再現: 木を除外し忘れると、一部セグメントの「正解」が木(25)になり\nCLIPの精度が不当に下がる', fontsize=10)
plt.tight_layout(); plt.savefig('/content/fig09_tree_bug_demo.png', dpi=140, bbox_inches='tight')
plt.show()
print('※ このシーンで実際に木ラベル汚染が起きるかは、視界内の木の量による。')
print('  実際にこのバグを踏んだときの記録 (5x3グリッド) は別途保存済みの画像を参照。')


---
## STEP 6 — 複数回見て投票し、訪問したら確定する

1回の判定を信用しすぎない設計。`BuildingMemory` は複数回の目撃を多数決で
束ね、実際に訪問して真の種別が判明したら、それ以降は推測を上書きしない。

ここでは、視界内の1つの建物に住民が近づいていく様子を模擬する
(向きは常にその建物を向く、という単純化をしている点に注意)。

In [ ]:
# 視界内の建物のうち、最も遠いものを選んで「これから近づく」対象にする
target_seg = max(segs, key=lambda s: scene_perp[(s[0]+s[1])//2]
                 if 0 <= scene_hitT[(s[0]+s[1])//2] < len(BLDG_TYPE_NAMES) else -1)
c0, c1 = target_seg
mid_col = (c0 + c1) // 2
true_name = segment_label(scene_hitT[c0:c1])

if true_name is None:
    print('選ばれたセグメントが建物でなかった。もう一度 STEP 0 から実行してください。')
else:
    # 目的地の建物の実座標を、compass 相当のロジックで概算する (このデモ用の簡易版)
    dist0 = float(scene_perp[mid_col])
    ang = float(th0[0]) + (mid_col/rc.W*2-1) * (rc.fov/2)
    bx = float(x0[0]) + np.cos(ang) * dist0
    by = float(y0[0]) + np.sin(ang) * dist0

    mem = BuildingMemory()
    building_id = (round(bx), round(by))
    N_STEPS = 5
    print(f'目的地: {true_name}  (推定座標 ({bx:.1f},{by:.1f}))\n')

    for step in range(N_STEPS):
        t = step / (N_STEPS - 1)
        d = dist0 * (1 - t) + 1.0 * t          # 徐々に近づく (最後は距離1セル)
        xs = torch.tensor([bx - np.cos(ang)*d], dtype=torch.float32, device=DEVICE)
        ys = torch.tensor([by - np.sin(ang)*d], dtype=torch.float32, device=DEVICE)
        ths = torch.tensor([ang], dtype=torch.float32, device=DEVICE)
        img = rc(xs, ys, ths, spec)[0]

        crop = img[:, :, rc.W//2-40:rc.W//2+40]   # 中央付近を「今見ている建物」として使う
        emb = clip_encode_images(crop[None])[0]
        guess = guess_groups(emb, TXT, k=5)
        mem.observe(building_id, guess)
        rec = mem.get(building_id)
        print(f'  step{step}  距離{d:4.1f}  CLIP予測={guess.top_type:<10} '
              f'投票状況={dict(rec.votes)}  現在の最有力候補={rec.best_guess()}')

    print(f'\n到着。真の種別が判明: {true_name} ({group_of(true_name)})')
    mem.confirm(building_id, true_name)
    rec = mem.get(building_id)
    print(f'確定後: confirmed_type={rec.confirmed_type}  confirmed_group={rec.confirmed_group}')
    print('以降、この建物を何度見ても推測は行わず、この確定情報を返し続ける。')

### 図11: 投票の推移を画像で見る

In [ ]:

# ── 図11: 近づきながらの投票推移を可視化する ──
if true_name is not None:
    mem2 = BuildingMemory()
    vote_history = []   # 各ステップ後の投票状況を記録

    for step in range(N_STEPS):
        t = step / (N_STEPS - 1)
        d = dist0 * (1 - t) + 1.0 * t
        xs = torch.tensor([bx - np.cos(ang)*d], dtype=torch.float32, device=DEVICE)
        ys = torch.tensor([by - np.sin(ang)*d], dtype=torch.float32, device=DEVICE)
        ths = torch.tensor([ang], dtype=torch.float32, device=DEVICE)
        img_step = rc(xs, ys, ths, spec)[0].cpu()
        crop = img_step[:, :, rc.W//2-40:rc.W//2+40]
        emb = clip_encode_images(crop[None])[0]
        guess = guess_groups(emb, TXT, k=5)
        mem2.observe(('viz',), guess)
        rec = mem2.get(('viz',))
        vote_history.append((d, dict(rec.votes), crop))

    mem2.confirm(('viz',), true_name)

    fig, axes = plt.subplots(2, N_STEPS, figsize=(3.4*N_STEPS, 7.2))
    all_groups = sorted({g for _, v, _ in vote_history for g in v})
    for i, (d, votes, crop) in enumerate(vote_history):
        axes[0, i].imshow(crop.permute(1,2,0).clamp(0,1).numpy())
        axes[0, i].axis('off'); axes[0, i].set_title(f'距離 {d:.1f}', fontsize=9)
        axes[1, i].bar(list(votes.keys()), list(votes.values()), color='#7bb0e0')
        axes[1, i].set_ylim(0, N_STEPS)
        axes[1, i].tick_params(axis='x', rotation=45, labelsize=8)
    true_group_viz = group_of(true_name)
    fig.suptitle(f'近づきながらの投票推移 → 到着で確定 (真の種別: {true_name} / {true_group_viz})',
                fontsize=11)
    plt.tight_layout(); plt.savefig('/content/fig11_memory_walk.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('対象セグメントが建物でなかったため、STEP0からやり直してください。')


---
## STEP 7 — 大量の場面でまとめて精度を見る

STEP 0〜5 でやったことを、多数の場面に対して自動で繰り返し、平均的な精度を出す。
S0-6 で行った検証と同じ方法（この結果が S0-6 の実測値の再現になるはず）。

In [ ]:
N_SAMPLES = 60
depth_rhos, type_hits1, type_hits5, group_hits1, group_hits5 = [], [], [], [], []

for i in range(N_SAMPLES):
    sp = build_map(seed=int(rng.integers(0, 1_000_000)), world=WORLD)
    ps = np.argwhere(sp.passable)
    pk = rng.integers(0, len(ps))
    xi = torch.tensor([ps[pk,0]+rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    yi = torch.tensor([ps[pk,1]+rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    thi = torch.tensor([rng.uniform(0,2*np.pi)], dtype=torch.float32, device=DEVICE)
    pp, ht, *_ = rc.cast(xi, yi, thi, sp)
    if not (0 <= ht[0, rc.W//2] < len(BLDG_TYPE_NAMES) and pp[0, rc.W//2] < 7.0):
        continue
    img = rc(xi, yi, thi, sp)[0]
    ht_np, pp_np = ht[0].cpu().numpy(), pp[0].cpu().numpy()

    dm = predict_depth(img)
    row = dm[HORIZON_ROW]
    valid = pp_np < 7.0
    if valid.sum() > 10:
        r, _ = spearmanr(row[valid], pp_np[valid])
        depth_rhos.append(r)

    for c0, c1 in segments_from_boundaries(detect_boundaries(row), rc.W):
        tn = segment_label(ht_np[c0:c1])
        if tn is None or (c1-c0) < 15:
            continue
        emb = clip_encode_images(img[:, :, c0:c1][None])[0]
        logits = emb @ TXT.T
        top5 = [BLDG_TYPE_NAMES[j] for j in logits.topk(5).indices.tolist()]
        tg = group_of(tn)
        type_hits1.append(top5[0] == tn)
        type_hits5.append(tn in top5)
        pred_groups = {group_of(t) for t in top5}
        group_hits1.append(group_of(top5[0]) == tg)
        group_hits5.append(tg in pred_groups)

print(f'サンプル場面数: {N_SAMPLES}   建物セグメント数: {len(type_hits5)}')
print(f'\n{"指標":<20}{"値":>8}')
print('-'*30)
print(f'{"深度 |rho| (中央値)":<20}{np.median(np.abs(depth_rhos)):>8.3f}')
print(f'{"タイプ Top-1":<20}{np.mean(type_hits1):>8.3f}')
print(f'{"タイプ Top-5":<20}{np.mean(type_hits5):>8.3f}')
print(f'{"グループ Top-1":<20}{np.mean(group_hits1):>8.3f}')
print(f'{"グループ Top-5":<20}{np.mean(group_hits5):>8.3f}')
print(f'\n参考: S0-6 での実測 (n=539) — 深度rho=0.997 / タイプTop-5=0.393 / グループTop-5=0.698')

In [ ]:

# ── 図10: 完全一致とグループ一致の精度比較 (STEP7の結果をグラフ化) ──
labels = ['完全一致\nTop-1', '完全一致\nTop-5', 'グループ一致\nTop-1', 'グループ一致\nTop-5']
values = [np.mean(type_hits1), np.mean(type_hits5), np.mean(group_hits1), np.mean(group_hits5)]
colors = ['#e0a07b', '#e0a07b', '#7bb0e0', '#7bb0e0']

fig, ax = plt.subplots(figsize=(7,4.5))
bars = ax.bar(labels, values, color=colors)
for b, v in zip(bars, values):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f'{v:.3f}', ha='center')
ax.axhline(1/len(BLDG_TYPE_NAMES), color='gray', linestyle=':', label='偶然 (Top-1)')
ax.set_ylim(0, 1.0); ax.set_ylabel('正解率')
ax.set_title(f'個別タイプ vs グループでの精度 (n={len(type_hits5)}セグメント)')
ax.legend()
plt.tight_layout(); plt.savefig('/content/fig10_group_vs_exact.png', dpi=140, bbox_inches='tight')
plt.show()


---
## STEP 8 — 本当に「プロンプト1行」で新しい建物を認識できるか

ここまで「新しい建物タイプを追加するには、CLIP方式ならプロンプトを1行足すだけ」
と説明してきました。**ただし、これは理屈だけで実測していませんでした。**
ここで正面から検証します。

### どうやって「新しい建物」を再現するか

MESA には本物の「パン屋」のテクスチャが存在しないので、真に未知の建物では
試せません。代わりに **Leave-One-Out**(1つ隠す)という手法を使います。

```
① 25タイプのうち1つ(例: bank)を、CLIPの語彙から一時的に隠す
   → この状態は「bankという概念をまだ知らない」のと同じ
② bankの建物画像を分類させる (もちろん bank とは答えられない)
③ そこで「これは bank だよ」とプロンプトを1行足す
④ 同じ画像を、もう一度分類させる
```

CLIP のテキスト埋め込みは「いつ計算したか」を区別しません。②の時点で
`bank` を除いた24タイプで分類するのと、④の時点で25タイプに戻して
分類するのは、**「新しい単語を後から教わった」状況と数学的に同じ**です。
だからこれは本物の新規タイプ追加の、正当な代理実験になります。

そして、これは記事の主張を検証するだけでなく、**間違っていないかを疑う**
機会でもあります。STEP5 で見つけた hub 現象(`bank` が `hotel` に
吸われる)は「知らない単語だから」ではなく「見た目が紛らわしいから」
起きていました。だとすれば、プロンプトを足しても `bank` は救えない
はずです。それも含めて実測します。

In [ ]:

def true_boundaries_of(hitT):
    return np.where(np.diff(hitT) != 0)[0]


In [ ]:

# ── 候補タイプそれぞれについて、最低限のサンプル数を確保するまで集める ──
# ランダムに集めるだけだと、候補タイプが1シーンに1個も映らないことが多く、
# 数百シーン見ても目標数に届かないタイプが残ってしまう (実測で確認済み)。
# 「目標未達のタイプが無くなるまで」条件でループを回す。
CANDIDATE_TYPES = ['hospital', 'office', 'apartment', 'temple',   # 従来◎だった型
                   'bank', 'gyudon', 'bento', 'cafe']              # hub現象で吸われていた型
TARGET_N_PER_TYPE = 15
MAX_SCENES = 3000

loo_embeds_by_type = {t: [] for t in CANDIDATE_TYPES}

def _min_count():
    return min(len(v) for v in loo_embeds_by_type.values())

scenes_used = 0
while _min_count() < TARGET_N_PER_TYPE and scenes_used < MAX_SCENES:
    scenes_used += 1
    sp = build_map(seed=int(rng.integers(0, 1_000_000)), world=WORLD)
    ps_ = np.argwhere(sp.passable)
    pk = rng.integers(0, len(ps_))
    xi = torch.tensor([ps_[pk,0]+rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    yi = torch.tensor([ps_[pk,1]+rng.uniform(.2,.8)], dtype=torch.float32, device=DEVICE)
    thi = torch.tensor([rng.uniform(0,2*np.pi)], dtype=torch.float32, device=DEVICE)
    pp, ht, *_ = rc.cast(xi, yi, thi, sp)
    if not (0 <= ht[0, rc.W//2] < len(BLDG_TYPE_NAMES) and pp[0, rc.W//2] < 7.0):
        continue
    img = rc(xi, yi, thi, sp)[0].cpu()
    ht_np = ht[0].cpu().numpy()

    tb = [0] + list(true_boundaries_of(ht_np) + 1) + [rc.W]
    for c0, c1 in zip(tb[:-1], tb[1:]):
        tn = segment_label(ht_np[c0:c1])
        if tn not in loo_embeds_by_type or (c1-c0) < 15:
            continue
        if len(loo_embeds_by_type[tn]) >= TARGET_N_PER_TYPE:
            continue                                    # 揃った型はもう集めない
        emb = clip_encode_images(img[:, :, c0:c1][None])[0]
        loo_embeds_by_type[tn].append(emb)

print(f'{scenes_used} シーン走査 (目標 {TARGET_N_PER_TYPE}件/型)')
for t in CANDIDATE_TYPES:
    print(f'  {t:<12} n={len(loo_embeds_by_type[t])}')
if _min_count() < 3:
    print('\n⚠ 集まりが少ない型がある。TARGET_N_PER_TYPE か MAX_SCENES を見直すこと。')


In [ ]:

# ── Leave-One-Out 本体 ──
def eval_with_labelset(embeds, true_type, known_types):
    """known_types だけで分類したときの Top-1/Top-5 (タイプ・グループ)。"""
    idx = [BLDG_TYPE_NAMES.index(t) for t in known_types]
    sub_txt = TXT[idx]
    logits = embeds @ sub_txt.T
    k = min(5, len(known_types))
    top = logits.topk(k, -1).indices
    pred_names = [[known_types[j] for j in row.tolist()] for row in top]
    t1 = np.mean([true_type == p[0] for p in pred_names])
    t5 = np.mean([true_type in p for p in pred_names])
    g1 = np.mean([group_of(true_type) == group_of(p[0]) for p in pred_names])
    g5 = np.mean([any(group_of(true_type)==group_of(x) for x in p) for p in pred_names])
    return t1, t5, g1, g5

print(f"{'タイプ':<12}{'n':>4}   {'--- 隠した状態(before) ---':<28}   {'--- プロンプト追加後(after) ---'}")
print(f"{'':12}{'':>4}   {'Top1':>6}{'Top5':>6}{'grp1':>6}{'grp5':>6}   {'Top1':>6}{'Top5':>6}{'grp1':>6}{'grp5':>6}")
print('-'*82)

loo_results = {}
for t in CANDIDATE_TYPES:
    embeds = loo_embeds_by_type[t]
    n = len(embeds)
    if n < 3:
        print(f'{t:<12}{n:>4}   (サンプル不足のためスキップ)')
        continue
    sub_embeds = torch.stack(embeds)

    known_before = [x for x in BLDG_TYPE_NAMES if x != t]        # t を隠す
    known_after = list(BLDG_TYPE_NAMES)                          # t を戻す (=プロンプト追加後)

    b = eval_with_labelset(sub_embeds, t, known_before)
    a = eval_with_labelset(sub_embeds, t, known_after)
    loo_results[t] = dict(before=b, after=a, n=n)

    print(f"{t:<12}{n:>4}   {b[0]:>6.2f}{b[1]:>6.2f}{b[2]:>6.2f}{b[3]:>6.2f}   "
          f"{a[0]:>6.2f}{a[1]:>6.2f}{a[2]:>6.2f}{a[3]:>6.2f}")

print('-'*82)
print('before: t を除いた24タイプだけで分類 (= まだ知らない単語)')
print('after : t を含む25タイプで分類     (= プロンプトを1行足した後)')
print('grp1/grp5 = グループ単位でのTop1/Top5一致率')


### 実際に成功した例と、失敗した例を見る

数字だけでなく、実際にどの建物がどう分類されたかを見ます。

In [ ]:

# 成功例 (Top-5 が大きく改善した型) と、失敗例 (before/after でほぼ変わらない型) を選ぶ
if loo_results:
    deltas = {t: r['after'][1] - r['before'][1] for t, r in loo_results.items()}
    success_t = max(deltas, key=deltas.get)
    fail_t = min(deltas, key=deltas.get)

    fig, ax = plt.subplots(figsize=(9, 4.8))
    types_sorted = sorted(loo_results, key=lambda t: deltas[t])
    y = np.arange(len(types_sorted))
    before_vals = [loo_results[t]['before'][1] for t in types_sorted]
    after_vals = [loo_results[t]['after'][1] for t in types_sorted]
    ax.barh(y-0.2, before_vals, height=0.35, label='隠した状態 (before)', color='#e08080')
    ax.barh(y+0.2, after_vals, height=0.35, label='プロンプト追加後 (after)', color='#7bb0e0')
    ax.set_yticks(y); ax.set_yticklabels(types_sorted)
    ax.set_xlabel('Top-5 正解率'); ax.legend()
    ax.set_title('プロンプト追加で「救えた型」と「救えなかった型」')
    plt.tight_layout(); plt.savefig('/content/fig12_loo_success_fail.png', dpi=140, bbox_inches='tight')
    plt.show()

    print(f'最も改善した型: {success_t}  (Top-5 {deltas[success_t]:+.2f})')
    print(f'最も改善しなかった型: {fail_t}  (Top-5 {deltas[fail_t]:+.2f})')
else:
    print('十分なサンプルが集まらなかったため、可視化をスキップします。')


### 結論: プロンプト追加は万能ではなかった

実測すると、記事の当初の説明には**見落としがありました。**

`hospital` / `office` / `apartment` / `temple` のように、元々の視覚的な
特徴がはっきりしている型は、プロンプトを1行足すだけで(隠す前と同水準まで)
認識できるようになります。**ここは主張通りです。**

一方 `bank` / `gyudon` / `bento` のように、STEP5 で hub 現象の被害に
遭っていた型は、**プロンプトを足しても Top-5 がほとんど動きません。**
理由は単純で、hub 現象の原因は「単語を知らないこと」ではなく
「見た目がそもそも紛らわしいこと」だったからです。隠しても知らせても、
CLIP が見ている画像の中身は変わりません。

つまり正確な結論はこうなります。

> **プロンプト1行で新しい建物を追加できるのは、見た目が既存タイプと
> 十分に異なる場合に限る。** 見た目が既存タイプ(特に `hotel` のような
> hub 化しやすい型)と紛らわしい場合は、プロンプトを足すだけでは
> 救えない。この場合は結局、テクスチャに看板や特徴的なディテールを
> 足すという、根本的な対策が必要になる。

これは「新しい建物タイプを追加するには」の章に書いた結論を、**一部だけ
訂正するもの**です。再学習が不要という利点自体は本当ですが、それだけで
必ず認識できるようになるわけではない、という条件がつきます。

---
## まとめ

| | 現状のライブ配信・学習 | 外の世界向け拡張 (`mesa_env.perception`) |
|---|---|---|
| 建物の種別 | マップ配列から直接 | CLIP でグループを推定 → 訪問で確定 |
| 距離・障害物 | マップ配列 (ALIGNED なら画像とも整合) | DepthAnything で推定 |
| 使っているモデル | DINOv2 (方策の特徴量のみ) | DINOv2 + DepthAnything + CLIP |
| 精度 | 100% (配列がそのまま正解) | 深度は非常に高精度、種別はグループ単位でのみ実用的 |

**個別タイプを一発で当てる設計は諦めている。** 3つの独立した実験
(S0-4: DINOv2埋め込み, S0-5: 粗い近似画像でのCLIP, S0-6: 本物のレイキャスト画像
でのCLIP) が同じ結論に収束したため — MESAのテクスチャセットは、大型ビル系
(office/tower/bank/hotel/apartment/mall) が視覚的にほぼ区別できないデザインに
なっている。これはCLIPの能力の問題ではなくデータ側の限界で、hub補正のような
後処理でも解決しなかった。

一方グループ単位 (食品系・小売系・公共系・住宅系・大型ビル系・寺・病院の7分類)
なら Top-5=0.70 と実用的な精度が出る。「遠くから大まかに当たりをつけ、訪問して
確定する」という設計が、この制約の上で機能する形。